# Week 3 · NumPy — Arrays and Vectorised Computation

**Programming for Data Science (606315)** · University of Petra

Pure-Python lists are flexible but slow and memory-hungry. NumPy's `ndarray` stores numbers in a
contiguous block of typed memory and runs loops in C. Every library you meet later —
Pandas, Scikit-learn, Matplotlib — is built on it, so the mental model you build here carries forward.

| Section | Topic |
|---|---|
| 1 | Why NumPy: speed and memory |
| 2 | Creating arrays |
| 3 | dtype, shape, and reshaping |
| 4 | Indexing, slicing, views vs copies |
| 5 | Boolean and fancy indexing |
| 6 | Vectorised arithmetic and **broadcasting** |
| 7 | Aggregations along an axis |
| 8 | Random numbers (the modern `Generator` API) |
| 9 | Linear algebra |
| 10 | A worked example: z-scores and outliers in real data |

In [ ]:
import numpy as np
from pathlib import Path
DATA = next(p / "data" / "generated" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "make_data.py").exists())
np.set_printoptions(precision=3, suppress=True)
print("NumPy", np.__version__)

## 1 · Why NumPy

Same computation, two ways. The speed-up ranges from a few × to over 100× depending on the operation and the machine, and it grows with the size of the data.

In [ ]:
import time, sys
n = 2_000_000
py_list = list(range(n))
arr = np.arange(n)

t0 = time.perf_counter(); _ = [x * 2.5 + 1 for x in py_list]; t_py = time.perf_counter() - t0
t0 = time.perf_counter(); _ = arr * 2.5 + 1;                  t_np = time.perf_counter() - t0
print(f"list comprehension: {t_py*1e3:7.1f} ms")
print(f"numpy             : {t_np*1e3:7.1f} ms   ({t_py/t_np:.0f}× faster)")
print(f"memory: list ≈ {sys.getsizeof(py_list)/1e6:.0f} MB (+ objects)   array = {arr.nbytes/1e6:.0f} MB")

The rule that follows from this: **never loop over array elements in Python if an array operation exists.**

## 2 · Creating arrays

In [ ]:
a = np.array([1, 2, 3])                      # from a list
b = np.array([[1.0, 2.0], [3.0, 4.0]])       # 2-D from nested lists
print(a, b, sep="\n")

print(np.zeros((2, 3)))                      # shape is a tuple
print(np.ones(4, dtype=int))
print(np.full((2, 2), 7))
print(np.eye(3))                             # identity
print(np.arange(0, 1, 0.25))                 # like range(), float step allowed
print(np.linspace(0, 1, 5))                  # 5 evenly spaced points, inclusive

## 3 · `dtype`, `shape`, `ndim` — the three attributes to always check

In [ ]:
m = np.arange(12)
print(m.dtype, m.shape, m.ndim)

m2 = m.reshape(3, 4)          # a *view* of the same memory, new shape
print(m2); print(m2.shape, m2.ndim)

print(m.reshape(2, -1).shape) # -1 means "work it out"
print(m2.T.shape)             # transpose
print(m2.ravel().shape)       # back to 1-D

In [ ]:
# dtype matters: integer division, overflow, and memory all depend on it
i = np.array([1, 2, 3])
f = i.astype(np.float32)
print(i / 2, i // 2, f.dtype, f.itemsize, "bytes per element")

small = np.array([250], dtype=np.uint8)
print(small + 10)             # wraps around! NumPy integers are fixed-width, unlike Python's

## 4 · Indexing and slicing — views, not copies

Slicing a NumPy array returns a **view**: modifying the slice modifies the original.
This is a deliberate design for performance, and the most common source of bugs for newcomers.

In [ ]:
x = np.arange(10)
print(x[2], x[-1], x[2:5], x[::2], x[::-1])

s = x[2:5]
s[:] = 99
print("original after editing the slice:", x)     # changed!

c = x[2:5].copy()
c[:] = -1
print("original after editing the copy :", x)     # unchanged

In [ ]:
g = np.arange(1, 13).reshape(3, 4)
print(g)
print("row 1        :", g[1])
print("element (1,2):", g[1, 2])        # row, column — one bracket, not g[1][2]
print("column 2     :", g[:, 2])
print("sub-block    :\n", g[0:2, 1:3])
print("last column  :", g[:, -1])

## 5 · Boolean masks and fancy indexing

A comparison on an array gives a boolean array of the same shape. Use it as an index to *filter*.
This replaces `if` statements inside loops and is the pattern Pandas uses for every `df[df.col > x]`.

In [ ]:
gpa = np.array([3.4, 2.1, 3.9, 2.8, 1.9, 3.2, 3.7])
mask = gpa >= 3.0
print(mask)
print(gpa[mask])
print(gpa[(gpa >= 2.0) & (gpa < 3.5)])     # combine with & | ~  — and use parentheses
print(np.where(gpa >= 3.0, "honours", "regular"))   # vectorised if/else
gpa[gpa < 2.0] = 2.0                        # assignment through a mask
print(gpa)

In [ ]:
# fancy indexing: index with an integer array → always a copy
names = np.array(["Ahmad", "Sara", "Omar", "Lina", "Khaled", "Noor", "Yousef"])
order = np.argsort(gpa)[::-1]               # indices that would sort descending
print(order)
print(names[order][:3], gpa[order][:3])     # top three

## 6 · Vectorised arithmetic and broadcasting

Arithmetic operates element-wise. When shapes differ, NumPy **broadcasts** the smaller array
across the larger one, following one rule: compare shapes from the right; dimensions must be
equal or one of them must be 1.

```
(3, 4)  +  (4,)   → (3, 4)     ✓  row vector added to every row
(3, 4)  +  (3, 1) → (3, 4)     ✓  column vector added to every column
(3, 4)  +  (3,)   → error      ✗  3 ≠ 4
```

In [ ]:
scores = np.array([[80, 90, 70],
                   [60, 85, 95],
                   [75, 65, 80]])          # 3 students × 3 exams
weights = np.array([0.3, 0.3, 0.4])        # one weight per exam  → shape (3,)

print(scores * weights)                    # broadcast across rows
print("weighted total per student:", (scores * weights).sum(axis=1))

bonus = np.array([[5], [0], [10]])         # one bonus per student → shape (3, 1)
print(scores + bonus)

In [ ]:
# Standardising columns is broadcasting in one line
col_mean = scores.mean(axis=0)             # shape (3,)
col_std  = scores.std(axis=0)
z = (scores - col_mean) / col_std
print(z)
print("column means after standardising:", z.mean(axis=0))

## 7 · Aggregations and the `axis` argument

`axis=0` collapses rows (gives one value **per column**); `axis=1` collapses columns (one value **per row**).
No `axis` collapses everything.

In [ ]:
print(scores.sum(), scores.mean(), scores.max())
print("per exam   (axis=0):", scores.mean(axis=0))
print("per student(axis=1):", scores.mean(axis=1))
print("argmax per student :", scores.argmax(axis=1))    # which exam was each student's best
print("cumulative          :", np.cumsum([1, 2, 3, 4]))
print("any/all             :", (scores > 90).any(), (scores > 50).all())

In [ ]:
# NaN-aware aggregations — real data has holes
with_nan = np.array([3.4, np.nan, 2.8, 3.9])
print(with_nan.mean(), np.nanmean(with_nan), np.isnan(with_nan).sum())

## 8 · Random numbers

Use the modern `np.random.default_rng(seed)` API, not the legacy `np.random.seed`.
A seeded generator makes experiments reproducible — essential for coursework and for research.

In [ ]:
rng = np.random.default_rng(42)
print(rng.integers(1, 7, size=5))                 # dice
print(rng.random(3))                              # uniform [0,1)
print(rng.normal(loc=2.9, scale=0.5, size=4))     # mean 2.9, sd 0.5
print(rng.choice(["DS", "CS", "AI"], size=5, p=[0.5, 0.3, 0.2]))
print(rng.permutation(6))

# the same seed reproduces the same draws
print(np.random.default_rng(42).integers(1, 7, size=5))

In [ ]:
# Simulation in two lines: the probability that the mean of 30 dice rolls exceeds 4
rolls = rng.integers(1, 7, size=(100_000, 30))
print((rolls.mean(axis=1) > 4).mean())

## 9 · Linear algebra

`@` is matrix multiplication; `*` is element-wise. `np.linalg` covers the rest.

In [ ]:
A = np.array([[2., 1.], [1., 3.]])
b = np.array([3., 5.])

print(A @ b)                              # matrix-vector product
print(A * A)                              # element-wise, NOT A²
print(np.linalg.inv(A) @ A)               # ≈ identity
x = np.linalg.solve(A, b)                 # solve Ax = b (prefer over inv)
print("x =", x, "check:", A @ x)
print("det", np.linalg.det(A), " norm", np.linalg.norm(b))
vals, vecs = np.linalg.eig(A)
print("eigenvalues", vals)

In [ ]:
# Least squares: fit y = m·x + c with one call — the core of linear regression
x = np.arange(10)
y = 2.5 * x + 1 + rng.normal(0, 1, 10)
X = np.column_stack([x, np.ones_like(x)])
(m, c), *_ = np.linalg.lstsq(X, y, rcond=None)
print(f"fitted slope {m:.2f}, intercept {c:.2f}")

## 10 · Worked example — z-scores and outliers in `sales_2025.csv`

We load one numeric column with `np.genfromtxt`, then do in NumPy what you will do in Pandas next week.
Notice there is not a single Python `for` loop.

In [ ]:
data = np.genfromtxt(DATA / "sales_2025.csv", delimiter=",", names=True, dtype=None, encoding="utf-8")
print(data.dtype.names, data.shape)

units = data["units"].astype(float)
revenue = units * data["unit_price"]

z = (revenue - revenue.mean()) / revenue.std()
outliers = np.abs(z) > 3
print(f"{outliers.sum()} outlier rows out of {len(revenue)}")
print("largest sale:", data[revenue.argmax()])

# revenue per product with a boolean mask per category
for p in np.unique(data["product"]):
    print(f"{p:11s} {revenue[data['product'] == p].sum():12,.0f}")

## Summary

* Arrays are **typed, fixed-shape, contiguous** — check `dtype`, `shape`, `ndim`.
* Slices are **views**; use `.copy()` when you need independence.
* **Boolean masks** replace `if` inside loops; combine with `&`, `|`, `~` and parentheses.
* **Broadcasting**: align shapes from the right; 1 stretches.
* `axis=0` → per column, `axis=1` → per row.
* `np.random.default_rng(seed)` for reproducible experiments.
* `@` for matrix products, `np.linalg.solve` instead of `inv`.

Next: `02_exercises.ipynb`.